# 中芯国际(688981.SH) 技术指标分析实验室

本 Notebook 对 **中芯国际(688981.SH)** 的日线数据进行技术指标分析，涵盖数据诊断、5 个指标的计算与可视化。

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 中文字体配置
plt.rcParams['font.sans-serif'] = ['SimSun']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 9

print('库加载完成')

## 1. 数据加载

In [ ]:
# 加载股价数据
DATA_PATH = 'data.csv'  # 修改为实际数据路径
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df['trade_date'] = pd.to_datetime(df['trade_date'].astype(str), format='%Y%m%d')
df = df.sort_values('trade_date').reset_index(drop=True)

n_rows = len(df)
date_start = df['trade_date'].min().date()
date_end = df['trade_date'].max().date()
print(f'数据加载完成: {n_rows} 行, 日期范围 {date_start} ~ {date_end}')

## 2. 数据诊断分析

### 2.1 缺失值检查

In [ ]:
# 缺失值统计
missing = df.isnull().sum()
has_missing = missing.sum() > 0
print('缺失值统计:')
if has_missing:
    display(pd.DataFrame({'字段': missing.index, '缺失数': missing.values}))
else:
    print('无缺失值')

### 2.2 描述性统计量

In [ ]:
# 核心字段描述性统计
desc_cols = ['open', 'high', 'low', 'close', 'vol', 'pct_chg']
desc = df[desc_cols].describe()
display(desc.round(2))

close_mean = df['close'].mean()
close_std = df['close'].std()
print(f'收盘价均值: {close_mean:.2f} 元, 标准差: {close_std:.2f} 元')

### 2.3 收盘价走势

In [ ]:
# 绘制收盘价走势
show = df.tail(250).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(show['trade_date'], show['close'], color='#d4380d', linewidth=1, label='收盘价')
ax.fill_between(show['trade_date'], show['close'].min()*0.95, show['close'], alpha=0.08, color='#d4380d')
ax.set_title('图1  收盘价走势（近250个交易日）', fontsize=10)
ax.set_xlabel('日期')
ax.set_ylabel('价格（元）')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()

## 3. RSI 相对强弱指标

In [ ]:
# RSI(14) 计算
def calc_rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['rsi_14'] = calc_rsi(df['close'])
rsi_val = df['rsi_14'].iloc[-1]
print(f'最新 RSI(14): {rsi_val:.2f}')

### RSI 相对强弱指标 可视化

In [ ]:
# RSI 可视化
show = df.tail(250).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), gridspec_kw=dict(height_ratios=[2, 1]), sharex=True)
ax1.plot(show['trade_date'], show['close'], color='#d4380d', linewidth=1)
ax1.set_ylabel('收盘价（元）')
ax1.set_title('图2  RSI(14) 相对强弱指标分析', fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(show['trade_date'], show['rsi_14'], color='#1890ff', linewidth=1, label='RSI(14)')
ax2.axhline(y=70, color='#ff4d4f', linestyle='--', linewidth=0.8, label='超买(70)')
ax2.axhline(y=30, color='#52c41a', linestyle='--', linewidth=0.8, label='超卖(30)')
ax2.fill_between(show['trade_date'], 70, 100, alpha=0.06, color='#ff4d4f')
ax2.fill_between(show['trade_date'], 0, 30, alpha=0.06, color='#52c41a')
ax2.set_ylabel('RSI'); ax2.set_ylim(0, 100)
ax2.legend(loc='upper left', fontsize=7, ncol=3)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout(); plt.show()

**解读要点：**
- RSI > 70 为超买区，价格可能回调；RSI < 30 为超卖区，价格可能反弹
- RSI = 50 为多空分界线
- RSI 与价格背离时，是趋势反转的重要信号
- 当前 RSI 值可从上方 Cell 输出中查看

## 4. MACD 指数平滑异同移动平均线

In [ ]:
# MACD(12,26,9) 计算
def calc_macd(close, fast=12, slow=26, signal=9):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    hist = (dif - dea) * 2
    return dif, dea, hist

df['dif'], df['dea'], df['macd_hist'] = calc_macd(df['close'])
dif_val = df['dif'].iloc[-1]; dea_val = df['dea'].iloc[-1]; macd_val = df['macd_hist'].iloc[-1]
print(f'最新 DIF: {dif_val:.2f}, DEA: {dea_val:.2f}, MACD柱: {macd_val:.2f}')

### MACD 指数平滑异同移动平均线 可视化

In [ ]:
# MACD 可视化
show = df.tail(250).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), gridspec_kw=dict(height_ratios=[2, 1]), sharex=True)
ax1.plot(show['trade_date'], show['close'], color='#d4380d', linewidth=1, label='收盘价')
ax1.plot(show['trade_date'], show['dif'], color='#1890ff', linewidth=0.8, label='DIF')
ax1.plot(show['trade_date'], show['dea'], color='#faad14', linewidth=0.8, label='DEA')
ax1.set_ylabel('价格'); ax1.legend(loc='upper left', fontsize=7, ncol=3)
ax1.set_title('图3  MACD(12,26,9) 指标分析', fontsize=10)
ax1.grid(True, alpha=0.3)

colors = ['#ff4d4f' if v>=0 else '#52c41a' for v in show['macd_hist']]
ax2.bar(show['trade_date'], show['macd_hist'], color=colors, width=1, label='MACD柱')
ax2.axhline(y=0, color='#333', linewidth=0.5)
ax2.set_ylabel('MACD柱'); ax2.legend(loc='upper left', fontsize=7)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout(); plt.show()

**解读要点：**
- DIF 上穿 DEA 为金叉（买入信号），下穿为死叉（卖出信号）
- 红色柱为多头动能，绿色柱为空头动能
- MACD 柱由正转负提示短期调整
- DIF 与价格的背离是趋势反转的早期信号

## 5. 布林带 Bollinger Bands

In [ ]:
# 布林带(20,2) 计算
def calc_boll(close, period=20, std_dev=2):
    mid = close.rolling(period).mean()
    std = close.rolling(period).std()
    upper = mid + std_dev * std
    lower = mid - std_dev * std
    width = (upper - lower) / mid
    pct_b = (close - lower) / (upper - lower)
    return mid, upper, lower, width, pct_b

df['bb_mid'], df['bb_upper'], df['bb_lower'], df['bb_width'], df['bb_pct_b'] = calc_boll(df['close'])
bb_u = df['bb_upper'].iloc[-1]; bb_m = df['bb_mid'].iloc[-1]; bb_l = df['bb_lower'].iloc[-1]
print(f'最新 上轨: {bb_u:.2f}, 中轨: {bb_m:.2f}, 下轨: {bb_l:.2f}')

### 布林带 Bollinger Bands 可视化

In [ ]:
# 布林带可视化
show = df.tail(250).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(show['trade_date'], show['close'], color='#d4380d', linewidth=1, label='收盘价')
ax.plot(show['trade_date'], show['bb_mid'], color='#1890ff', linewidth=0.8, linestyle='--', label='中轨')
ax.plot(show['trade_date'], show['bb_upper'], color='#722ed1', linewidth=0.8, label='上轨')
ax.plot(show['trade_date'], show['bb_lower'], color='#52c41a', linewidth=0.8, label='下轨')
ax.fill_between(show['trade_date'], show['bb_upper'], show['bb_lower'], alpha=0.05, color='#722ed1')
ax.set_title('图4  布林带(20,2) 指标分析', fontsize=10)
ax.set_ylabel('价格（元）'); ax.legend(loc='upper left', fontsize=7, ncol=4)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout(); plt.show()

**解读要点：**
- 价格触及上轨可能超买，触及下轨可能超卖
- 布林带收窄预示低波动，可能即将突破
- 布林带张开预示波动增大，趋势加速
- %B 指标量化价格在布林带中的相对位置

## 6. ATR 平均真实波幅

In [ ]:
# ATR(14) 计算
def calc_atr(high, low, close, period=14):
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()
    return tr, atr

df['tr'], df['atr_14'] = calc_atr(df['high'], df['low'], df['close'])
df['stop_loss'] = df['close'] - 2 * df['atr_14']
atr_val = df['atr_14'].iloc[-1]; sl_val = df['stop_loss'].iloc[-1]
print(f'最新 ATR(14): {atr_val:.2f}, 止损价: {sl_val:.2f}')

### ATR 平均真实波幅 可视化

In [ ]:
# ATR 可视化
show = df.tail(250).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), gridspec_kw=dict(height_ratios=[2, 1]), sharex=True)
ax1.plot(show['trade_date'], show['close'], color='#d4380d', linewidth=1)
ax1.plot(show['trade_date'], show['stop_loss'], color='#faad14', linewidth=0.8, linestyle='--', label='止损线')
ax1.set_ylabel('价格（元）'); ax1.legend(loc='upper left', fontsize=7)
ax1.set_title('图5  ATR(14) 波动率与止损分析', fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(show['trade_date'], show['atr_14'], color='#1890ff', linewidth=1, label='ATR(14)')
ax2.set_ylabel('ATR'); ax2.legend(loc='upper left', fontsize=7)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout(); plt.show()

**解读要点：**
- ATR 衡量市场波动率，数值越大波动越剧烈
- 止损线 = 收盘价 - 2×ATR，可作为动态止损参考
- ATR 上升时建议加宽止损间距，下降时可收紧

## 7. KDJ 随机指标

In [ ]:
# KDJ(9,3,3) 计算
def calc_kdj(high, low, close, n=9, m1=3, m2=3):
    low_min = low.rolling(n, min_periods=1).min()
    high_max = high.rolling(n, min_periods=1).max()
    rsv = (close - low_min) / (high_max - low_min) * 100
    rsv = rsv.fillna(50)
    k = rsv.ewm(alpha=1/m1, adjust=False).mean()
    d = k.ewm(alpha=1/m2, adjust=False).mean()
    j = 3 * k - 2 * d
    return k, d, j

df['k'], df['d'], df['j'] = calc_kdj(df['high'], df['low'], df['close'])
k_val = df['k'].iloc[-1]; d_val = df['d'].iloc[-1]; j_val = df['j'].iloc[-1]
print(f'最新 K: {k_val:.2f}, D: {d_val:.2f}, J: {j_val:.2f}')

### KDJ 随机指标 可视化

In [ ]:
# KDJ 可视化
show = df.tail(250).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), gridspec_kw=dict(height_ratios=[2, 1.5]), sharex=True)
ax1.plot(show['trade_date'], show['close'], color='#d4380d', linewidth=1)
ax1.set_ylabel('收盘价（元）')
ax1.set_title('图6  KDJ(9,3,3) 随机指标分析', fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(show['trade_date'], show['k'], color='#1890ff', linewidth=1, label='K线')
ax2.plot(show['trade_date'], show['d'], color='#faad14', linewidth=1, label='D线')
ax2.plot(show['trade_date'], show['j'], color='#eb2f96', linewidth=0.8, linestyle='--', label='J线')
ax2.axhline(y=80, color='#ff4d4f', linestyle=':', linewidth=0.6)
ax2.axhline(y=20, color='#52c41a', linestyle=':', linewidth=0.6)
ax2.fill_between(show['trade_date'], 80, 100, alpha=0.05, color='#ff4d4f')
ax2.fill_between(show['trade_date'], 0, 20, alpha=0.05, color='#52c41a')
ax2.set_ylabel('KDJ值'); ax2.set_ylim(-10, 110)
ax2.legend(loc='upper left', fontsize=7, ncol=3)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout(); plt.show()

**解读要点：**
- K 线上穿 D 线且在 20 以下为低位金叉（强买信号）
- K 线下穿 D 线且在 80 以上为高位死叉（强卖信号）
- J 值 > 100 超买，J 值 < 0 超卖
- KDJ 与 RSI 互补使用效果更好

## 8. 综合信号汇总

In [ ]:
# 综合信号汇总表 — 最近 10 个交易日
signal_df = df.tail(10).copy()

# 生成信号判断
signals = {}
if 'rsi_14' in signal_df.columns:
    signals['RSI值'] = signal_df['rsi_14'].round(2)
    signals['RSI信号'] = signal_df['rsi_14'].apply(
        lambda x: '超买' if x > 70 else ('超卖' if x < 30 else '中性'))
if 'macd_hist' in signal_df.columns:
    signals['MACD柱'] = signal_df['macd_hist'].round(2)
    signals['MACD信号'] = signal_df['macd_hist'].apply(
        lambda x: '多头' if x > 0 else '空头')
if 'bb_pct_b' in signal_df.columns:
    signals['布林%B'] = signal_df['bb_pct_b'].round(2)
    signals['布林信号'] = signal_df['bb_pct_b'].apply(
        lambda x: '超买' if x > 1 else ('超卖' if x < 0 else '中性'))
if 'atr_14' in signal_df.columns:
    signals['ATR'] = signal_df['atr_14'].round(2)
    signals['止损价'] = signal_df['stop_loss'].round(2)
if 'k' in signal_df.columns:
    signals['K'] = signal_df['k'].round(2)
    signals['D'] = signal_df['d'].round(2)
    signals['J'] = signal_df['j'].round(2)

summary = pd.DataFrame(signals, index=signal_df['trade_date'].dt.strftime('%Y-%m-%d'))
display(summary)

## 9. 结论与展望

本 Notebook 完成了 中芯国际(688981.SH) 的技术指标分析，涵盖 5 个经典指标的计算与可视化。

**下一步建议：**
- 组合多个指标信号进行综合判断，避免单一指标误判
- 对不同参数组合进行回测，验证信号有效性
- 结合基本面分析，提升决策质量
- 扩展到多股票横向对比分析